# 期末大作业数据环境

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
# Reading ratings file
ratings = pd.read_csv('./data/ratings2.csv',  encoding='latin-1', usecols=['user_id', 'movie_id', 'rating', 'timestamp'])

# Reading users file
users = pd.read_csv('./data/users.csv', encoding='latin-1', usecols=['user_id', 'gender', 'zipcode', 'age_desc', 'occ_desc'])

# Reading movies file
movies = pd.read_csv('./data/movies.csv',  encoding='latin-1', usecols=['movie_id', 'title', 'genres'])

# Reading movies info file
movies_info = pd.read_csv('./data/info.csv',  encoding='latin-1', usecols=['id', 'name', 'genre','intro','directors','starts', 'release_time'])
movies_info.rename(columns ={ 'id':'movie_id', 'starts': 'stars'}, inplace = True)

In [ ]:
movies 

## 电影海报向量化

In [ ]:
!pip install img2vec_pytorch -i https://pypi.tuna.tsinghua.edu.cn/simple
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import os
import cv2
import pandas as pd
import numpy as np
from PIL import Image
from img2vec_pytorch import Img2Vec

# 文件夹路径
folder_path = "./poster"

# 初始化特征列表
features = []

# 初始化Img2Vec模型（默认使用ResNet-50）
img2vec_model = Img2Vec(cuda=False)  # cuda=False使用CPU，如有GPU可设为True

# 遍历文件夹中的每个海报图像
for filename in os.listdir(folder_path):
    # 只处理jpg和png格式的图像文件
    if filename.endswith(".jpg") or filename.endswith(".png"):
        # 构建完整的图像路径
        image_path = os.path.join(folder_path, filename)
        
        # 使用PIL读取图像（Img2Vec需要PIL Image格式）
        image = Image.open(image_path)
        
        # 确保图像是RGB格式（Img2Vec要求3通道RGB图像）
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # 使用预训练模型生成图像的特征向量（维度通常为512或更多）
        vector = img2vec_model.get_vec(image)  # 返回numpy数组或tensor
        
        # 将向量转换为numpy数组（便于后续处理）
        feature_vector = vector.cpu().numpy() if hasattr(vector, 'cpu') else vector
        
        # 提取电影ID（去掉文件扩展名）
        movie_id = filename.split('.')[0]
        
        # 添加图像文件前缀（电影ID）和特征向量到特征列表中
        features.append([movie_id, feature_vector])
        
        # print(f"已处理: {movie_id}")  # 显示处理进度

# 将特征列表转换为DataFrame，便于管理和查询
columns = ['movie_id', 'features']
df = pd.DataFrame(features, columns=columns)

# 输出带有电影ID和特征向量的DataFrame
print("\nDataFrame with Movie ID and Features:")
# print(df.head())
print(f"\n总共处理了 {len(df)} 张海报图像")
print(f"每个特征向量的维度: {df.iloc[0]['features'].shape}")
print("有少数电影海报缺失，作业中可以排除")

## 基于向量空间的KNN检索

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# 使用系统支持的中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']  # 指定默认字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题


# ============================================
# 图像相似度查询与可视化功能
# ============================================

def find_similar_images(movie_id, df, img2vec_model, folder_path, top_k=5):
    """
    查找与指定电影海报最相似的K张海报
    
    参数:
    - movie_id: 要查询的电影ID（不含扩展名）
    - df: 包含所有电影特征向量的DataFrame
    - img2vec_model: 预训练的Img2Vec模型（用于新图像，这里实际不需要）
    - folder_path: 图像文件夹路径
    - top_k: 返回最相似图像的数量（不包括查询图像本身）
    
    返回:
    - similar_images: 包含(相似度, 电影ID)的列表
    - query_vector: 查询图像的特征向量
    """
    
    # 1. 查找查询图像的特征向量
    query_row = df[df['movie_id'] == movie_id]
    
    if len(query_row) == 0:
        print(f"错误：找不到电影ID '{movie_id}'")
        return None, None
    
    query_vector = query_row.iloc[0]['features'].reshape(1, -1)  # 重塑为2D数组
    
    # 2. 计算查询向量与所有其他图像向量的余弦相似度
    similarities = []
    for idx, row in df.iterrows():
        if row['movie_id'] != movie_id:  # 跳过查询图像本身
            # 计算余弦相似度
            target_vector = row['features'].reshape(1, -1)
            similarity = cosine_similarity(query_vector, target_vector)[0][0]
            similarities.append((similarity, row['movie_id']))
    
    # 3. 按相似度降序排序，获取Top-K
    similarities.sort(key=lambda x: x[0], reverse=True)
    similar_images = similarities[:top_k]
    
    return similar_images, query_vector


def visualize_similar_images(movie_id, similar_images, folder_path):
    """
    可视化查询图像及其最相似的K张图像
    
    参数:
    - movie_id: 查询的电影ID
    - similar_images: 包含(相似度, 电影ID)的列表
    - folder_path: 图像文件夹路径
    """
    
    # 计算需要显示的子图数量（1个查询图 + K个相似图）
    k = len(similar_images)
    total_images = 1 + k
    
    # 创建子图布局：1行，total_images列
    fig, axes = plt.subplots(1, total_images, figsize=(4*total_images, 4))
    
    # 确保axes是一维数组（当total_images=1时也能正常工作）
    if total_images == 1:
        axes = [axes]
    
    # 1. 显示查询图像
    query_path = os.path.join(folder_path, f"{movie_id}.jpg")
    if not os.path.exists(query_path):
        # 尝试png格式
        query_path = os.path.join(folder_path, f"{movie_id}.png")
    
    if os.path.exists(query_path):
        query_img = Image.open(query_path)
        axes[0].imshow(query_img)
        axes[0].set_title(f"查询图像\nID: {movie_id}", fontsize=12, fontweight='bold')
        axes[0].axis('off')
    else:
        axes[0].text(0.5, 0.5, f"图像不存在\nID: {movie_id}", 
                    ha='center', va='center', transform=axes[0].transAxes)
        axes[0].axis('off')
    
    # 2. 显示相似图像
    for i, (similarity, sim_id) in enumerate(similar_images, start=1):
        # 构建图像路径
        sim_path = os.path.join(folder_path, f"{sim_id}.jpg")
        if not os.path.exists(sim_path):
            sim_path = os.path.join(folder_path, f"{sim_id}.png")
        
        if os.path.exists(sim_path):
            sim_img = Image.open(sim_path)
            axes[i].imshow(sim_img)
            axes[i].set_title(f"相似度: {similarity:.3f}\nID: {sim_id}", 
                             fontsize=10)
            axes[i].axis('off')
        else:
            axes[i].text(0.5, 0.5, f"图像不存在\nID: {sim_id}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].axis('off')
    
    # 添加总标题
    plt.suptitle(f"电影海报相似度搜索 - 最相似的{len(similar_images)}个结果", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def interactive_search(df, img2vec_model, folder_path):
    """
    交互式搜索功能：让用户输入电影ID并显示相似图像
    
    参数:
    - df: 包含所有电影特征向量的DataFrame
    - img2vec_model: Img2Vec模型
    - folder_path: 图像文件夹路径
    """
    
    print("\n" + "="*60)
    print("电影海报相似度搜索系统")
    print("="*60)
    
    # 显示可用的电影ID列表（前20个）
    available_ids = df['movie_id'].tolist()
    print(f"\n可用的电影ID（共{len(available_ids)}个）: {available_ids[:20]}")
    if len(available_ids) > 20:
        print("... (更多ID请查看完整列表)")
    
    while True:
        print("\n" + "-"*60)
        movie_id = input("请输入要查询的电影ID（输入 'quit' 或 'q' 退出）: ").strip()
        
        if movie_id.lower() in ['quit', 'q']:
            print("感谢使用！再见！")
            break
        
        if movie_id not in available_ids:
            print(f"错误：找不到电影ID '{movie_id}'，请重新输入！")
            continue
        
        # 查找相似图像
        print(f"\n正在查找与 '{movie_id}' 最相似的图像...")
        similar_images, query_vector = find_similar_images(movie_id, df, img2vec_model, folder_path, top_k=5)
        
        if similar_images:
            print(f"\n找到 {len(similar_images)} 个相似图像：")
            for i, (sim, sim_id) in enumerate(similar_images, 1):
                print(f"  {i}. {sim_id} (相似度: {sim:.4f})")
            
            # 可视化结果
            visualize_similar_images(movie_id, similar_images, folder_path)

# 启动交互式搜索
interactive_search(df, img2vec_model, folder_path)
